# TDS Trend & Forecast — LAGWRP (Los Angeles–Glendale Water Reclamation Plant)

**Objetivo do projeto:** avaliar se a salinidade (TDS) do efluente da LAGWRP aumentou ao longo de ~15 anos (2011-2026), quantificar essa tendência, construir modelos de previsão para +10/+15/+20 anos, e investigar a correlação entre TDS e dois indicadores de desempenho do tratamento biológico (Amônia e BOD).

Este notebook reúne o projeto inteiro (pré-processamento, análise exploratória, bateria de métodos, comparação final) chamando as mesmas funções dos scripts `.py` na raiz do projeto — nunca duplicando a lógica manualmente. Ver `plano_projeto_TDS.md` para o plano completo.

**Status atual: projeto completo.** `script_00` a `script_15` implementados e validados — 21 métodos de previsão, todos com validação temporal honesta; análise de estrutura da série; correlação TDS-Amônia/BOD nos três tratamentos de ND; diagnóstico de resíduos dos candidatos mais fortes; comparação qualitativa com a literatura (fontes verificadas via WebFetch, não citadas às cegas); e síntese final com três finalistas complementares recomendados (regressão bayesiana, Detrend+RF, híbrido SARIMA+Prophet). Os quatro objetivos do projeto (tendência, previsão, correlação, interpretação em contexto real) estão endereçados.

> **Rastreamento de experimentos:** as execuções dos scripts são rastreadas localmente com MLflow (sem nuvem, sem conta — ver `plano_projeto_TDS.md` §4.3 e `utils/experiment_tracking.py`). `script_01` a `script_13` (exceto `script_00`/`script_06`, que não são metodologias de previsão) abrem uma run por método (rodando `python script_0X_....py`, não as chamadas diretas de função usadas neste notebook). Para abrir o dashboard: `mlflow ui --backend-store-uri sqlite:///mlflow.db`. O `resultados_comparacao.csv` continua sendo a fonte usada pelas células abaixo e pelo artigo — o MLflow é o histórico complementar de execuções (incluindo tentativas descartadas).

## 1. Carregamento e pré-processamento dos dados

Os dados brutos vêm do portal eSMR (California Water Boards), exportados em 4 planilhas (`TDS.csv`, `Chloride.csv`, `Ammonia.csv`, `BOD.csv`, `sep=';' decimal=','`). `script_00_preprocessamento.py` filtra a série mensal canônica do efluente (`EFF-001`+`EFF-001A` unificados, `Calculated Method == "Monthly Average (Mean)"`, `Units == "mg/L"`).

Os valores não detectados (ND) de BOD (65% dos meses, ver `plano_projeto_TDS.md` §1.3 e `script_00b_analise_censura_bod.py`) foram tratados de **três formas em paralelo** (decisão do usuário, aprovada após apresentação dos números reais):
- **Dataset A** — ND do BOD = MDL/2 (1,5 mg/L)
- **Dataset B** — ND do BOD = 0
- **Dataset F** — ND do BOD = estimativa ROS/Helsel (2,517 mg/L, ajustada a partir da forma real da distribuição dos valores detectados — ver `script_00b`)

In [ ]:
import sys
sys.path.insert(0, '.')
from script_00_preprocessamento import construir_datasets

dataset_a, dataset_b, dataset_c, base = construir_datasets()
print(f"Dataset A (BOD ND=MDL/2):      {dataset_a.shape}")
print(f"Dataset B (BOD ND=0):          {dataset_b.shape}")
print(f"Dataset F (BOD ND=ROS/Helsel): {dataset_c.shape}")
dataset_a.head()

In [2]:
print(f"Periodo: {base['Data'].min().date()} a {base['Data'].max().date()}  ({len(base)} meses)")
print(f"Meses com BOD ND: {int(base['BOD_foi_ND'].sum())} / {int(base['BOD_foi_ND'].notna().sum())}")
print()
print("Valores ausentes por coluna (dataset A):")
print(dataset_a.isna().sum())

Periodo: 2011-02-28 a 2026-03-31  (182 meses)
Meses com BOD ND: 118 / 181

Valores ausentes por coluna (dataset A):
Data            0
TDS_mgL         0
Chloride_mgL    0
Ammonia_mgL     0
BOD_foi_ND      1
BOD_mgL         1
dtype: int64


## 2. Análise exploratória

Séries mensais de TDS, Cloreto e Amônia (idênticas nos dois datasets) e BOD sob os dois tratamentos de ND.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 1, figsize=(9, 10), sharex=True)

axes[0].plot(dataset_a['Data'], dataset_a['TDS_mgL'], color='tab:blue')
axes[0].set_ylabel('TDS (mg/L)')
axes[0].set_title('Series mensais canonicas - efluente EFF-001 (unificado)')

axes[1].plot(dataset_a['Data'], dataset_a['Chloride_mgL'], color='tab:green')
axes[1].set_ylabel('Cloreto (mg/L)')

axes[2].plot(dataset_a['Data'], dataset_a['Ammonia_mgL'], color='tab:orange')
axes[2].set_ylabel('Amonia (mg/L)')

axes[3].plot(dataset_a['Data'], dataset_a['BOD_mgL'], color='tab:red', label='Dataset A (ND=MDL/2)', alpha=0.8)
axes[3].plot(dataset_b['Data'], dataset_b['BOD_mgL'], color='tab:purple', label='Dataset B (ND=0)', alpha=0.6, linestyle='--')
axes[3].plot(dataset_c['Data'], dataset_c['BOD_mgL'], color='tab:brown', label='Dataset F (ND=ROS/Helsel)', alpha=0.6, linestyle=':')
axes[3].set_ylabel('BOD (mg/L)')
axes[3].legend(fontsize=8)
axes[3].set_xlabel('Data')

plt.tight_layout()
plt.savefig('notebook_series_exploratorias.png', dpi=110)
plt.show()

In [ ]:
import numpy as np

print("Correlacao de Pearson entre parametros (Dataset A, ND=MDL/2):")
print(dataset_a[['TDS_mgL', 'Chloride_mgL', 'Ammonia_mgL', 'BOD_mgL']].corr().round(3))
print()
print("Correlacao de Pearson entre parametros (Dataset B, ND=0):")
print(dataset_b[['TDS_mgL', 'Chloride_mgL', 'Ammonia_mgL', 'BOD_mgL']].corr().round(3))
print()
print("Correlacao de Pearson entre parametros (Dataset F, ND=ROS/Helsel):")
print(dataset_c[['TDS_mgL', 'Chloride_mgL', 'Ammonia_mgL', 'BOD_mgL']].corr().round(3))

> **Nota metodológica:** as correlações acima são exploratórias (Pearson simples, sem controle de tendência temporal comum). A análise formal do objetivo 3 (com remoção de tendência e defasagem) está na próxima célula, via `script_06_correlacao_tds_amonia_bod.py`.

## 3.e Correlação TDS–Amônia e TDS–BOD (`script_06_correlacao_tds_amonia_bod.py`)

Implementado. Correlação de Pearson/Spearman bruta e destendenciada (resíduos de OLS vs. tempo, para isolar covariação real de tendência comum), com correlação cruzada em defasagens de 0 a 3 meses. Roda nos três datasets de tratamento de ND do BOD (A, B, F).

In [ ]:
from script_06_correlacao_tds_amonia_bod import construir_datasets as construir_datasets_06, analisar_par

d06_a, d06_b, d06_c, _ = construir_datasets_06()
d06_a['t_anos'] = (d06_a['Data'] - d06_a['Data'].iloc[0]).dt.days / 365.25
d06_b['t_anos'] = (d06_b['Data'] - d06_b['Data'].iloc[0]).dt.days / 365.25
d06_c['t_anos'] = (d06_c['Data'] - d06_c['Data'].iloc[0]).dt.days / 365.25

r_amonia = analisar_par('tds_amonia', d06_a, 'TDS_mgL', 'Ammonia_mgL')
r_bod_a = analisar_par('tds_bod_mdl2', d06_a, 'TDS_mgL', 'BOD_mgL')
r_bod_b = analisar_par('tds_bod_zero', d06_b, 'TDS_mgL', 'BOD_mgL')
r_bod_c = analisar_par('tds_bod_ros', d06_c, 'TDS_mgL', 'BOD_mgL')

for r in [r_amonia, r_bod_a, r_bod_b, r_bod_c]:
    print(f"{r['par']}: bruta r={r['bruta_pearson_r']:.3f} (p={r['bruta_pearson_p']:.4f})  "
          f"destend. r={r['destend_pearson_r']:.3f} (p={r['destend_pearson_p']:.4f})")

**TDS vs. Amônia:** correlação positiva e significativa mesmo após remover a tendência comum (r=0,175, p=0,018; mais forte com 1 mês de defasagem, r=0,215) — consistente com a hipótese de que salinidade mais alta está associada a nitrificação menos eficiente.

**TDS vs. BOD:** sem correlação significativa em nenhum dos três tratamentos de ND (A, B, F/ROS-Helsel) — inclusive no ROS/Helsel, o tratamento estatisticamente mais bem embasado dos três (r destendenciado -0,108, p=0,147). Isso reforça que a ausência de sinal é uma característica real da série de BOD (rente ao limite de detecção durante quase todo o período), não um artefato da escolha de substituição do ND. Ver `Artigo/src/resultados.tex` §Correlação para a discussão completa e `Artigo/src/conclusao.tex` para a interpretação em contexto real (objetivo 4), incluindo a referência ao mecanismo de Schwabe et al. (2020) sobre conservação de água e salinidade de esgoto.

## 2.5 Análise de estrutura da série (`script_07_analise_estrutura_serie.py`)

Implementado. Antes de aprofundar a bateria de modelagem, testamos a estrutura da série de TDS: força de sazonalidade (STL, estatística de Wang et al.), estacionariedade (ADF + KPSS, hipóteses opostas por desenho) e quebra estrutural (Chow no breakpoint aproximado de 2012, Pettitt sem breakpoint pré-definido, CUSUM). Nenhuma dessas propriedades é assumida — são testadas.

In [ ]:
from script_07_analise_estrutura_serie import (
    carregar_serie_tds as carregar_serie_tds_07, decompor_stl, testar_estacionariedade, testar_quebra_estrutural,
)

s_07 = carregar_serie_tds_07()
d_07 = pd.DataFrame({'Data': s_07.index, 'TDS_mgL': s_07.values})
d_07['t_anos'] = (d_07['Data'] - d_07['Data'].iloc[0]).dt.days / 365.25

stl_07, f_sazonal, f_tendencia = decompor_stl(s_07)
estac_nivel = testar_estacionariedade(s_07.values)
quebra_07 = testar_quebra_estrutural(d_07)

print(f"Forca de sazonalidade (Fs): {f_sazonal:.3f}  |  Forca de tendencia (Ft): {f_tendencia:.3f}")
print(f"ADF (nivel): stat={estac_nivel['adf_stat']:.3f} p={estac_nivel['adf_pvalor']:.4f}")
print(f"KPSS (nivel): stat={estac_nivel['kpss_stat']:.3f} p={estac_nivel['kpss_pvalor']:.4f}")
print(f"Chow test (breakpoint {quebra_07['chow_breakpoint_data']}): F={quebra_07['chow_f_stat']:.3f} p={quebra_07['chow_pvalor']:.4f}")
print(f"Pettitt test: mudanca em {quebra_07['pettitt_data_mudanca']} (p={quebra_07['pettitt_pvalor']:.4f}, dentro de janela de seca CA: {quebra_07['pettitt_dentro_janela_seca_ca']})")
print(f"CUSUM: stat={quebra_07['cusum_stat']:.3f} p={quebra_07['cusum_pvalor']:.4f}")

**Sazonalidade fraca/moderada** (Fs=0,25, abaixo do limiar de referência 0,64 da literatura de forecasting) — não assumida por padrão, mas os termos sazonais do SARIMA (§3.b) e do ETS (§3.f) continuam justificados pelo AIC/pela própria implementação, não descartados só por isso. **ADF rejeita não-estacionariedade e KPSS não rejeita estacionariedade** — os dois testes concordam que a série em nível é compatível com um processo estacionário em torno de uma tendência determinística (trend-stationary), não uma caminhada aleatória pura — contexto relevante para interpretar por que SARIMA (que diferencia a série) diverge tanto dos métodos de tendência linear. **Quebra estrutural confirmada**: Chow (breakpoint ~2012, coincide com a troca EFF-001→EFF-001A) e Pettitt (mudança detectada em 2014-04, dentro da janela de seca da Califórnia 2012-2016) e CUSUM concordam que a série não é homogênea ao longo do período — reforça a cautela já registrada na metodologia sobre extrapolar além do histórico. Ver `Artigo/src/resultados.tex` para a discussão completa.

## 3. Bateria de metodologias

Cada subseção abaixo corresponde a um `script_XX` da bateria (ver `plano_projeto_TDS.md` §3-4). Serão preenchidas com resultados reais assim que cada script for implementado e executado — **nenhum resultado é inventado aqui**.

### 3.a Estatística clássica de tendência (`script_01_mann_kendall_theilsen.py`)

Implementado. Roda sobre a série de TDS (univariada — resultado independe do tratamento de ND do BOD, ver nota no topo do script). Mann-Kendall+Sen e Theil-Sen, com holdout dos últimos 24 meses.

In [5]:
from script_01_mann_kendall_theilsen import (
    carregar_serie_tds, rodar_mann_kendall_sen, rodar_theil_sen, rodar_ols, HOLDOUT_MESES
)

d_tds = carregar_serie_tds()
treino = d_tds.iloc[:-HOLDOUT_MESES].reset_index(drop=True)
holdout = d_tds.iloc[-HOLDOUT_MESES:].reset_index(drop=True)

resultados_tendencia = [
    rodar_mann_kendall_sen(d_tds, treino, holdout),
    rodar_theil_sen(d_tds, treino, holdout),
    rodar_ols(d_tds, treino, holdout),
]

import pandas as pd
tabela = pd.DataFrame(resultados_tendencia)[
    ['metodo', 'tendencia_mgL_ano', 'tendencia_pvalor', 'rmse_holdout', 'mae_holdout', 'r2_holdout']
]
tabela

,metodo,tendencia_mgL_ano,tendencia_pvalor,rmse_holdout,mae_holdout,r2_holdout
0,mann_kendall_sen,3.906417,0.005607,51.052208,40.723430,-0.657860
1,theil_sen,3.906417,0.005586,51.052208,40.723430,-0.657860
2,ols,3.743707,0.004725,49.235822,39.367692,-0.541988


Os três métodos convergem para uma tendência de alta estatisticamente significativa (p < 0,01), de 3,7 a 3,9 mg/L/ano. O R² negativo no holdout indica que a reta de tendência simples, ajustada só ao treino, erra mais nos últimos 24 meses do que a média do treino — a série tem flutuações de curto prazo que uma tendência linear não captura, mesmo acertando a direção de longo prazo. Ver `Artigo/src/resultados.tex` para a discussão completa e a figura com a extrapolação +10/+15/+20 anos.

### 3.b Séries temporais clássicas — STL + ARIMA/SARIMA (`script_02_arima_sarima.py`)

Implementado. Também univariado em TDS (independe do tratamento de ND do BOD). STL extrai a tendência dessazonalizada; SARIMA tem a ordem escolhida por busca em grade (AIC), com `d=1` e `D` explorado em {0,1} — drift explícito só é permitido quando `D=0`, para evitar tendência quadrática espúria com dupla diferenciação.

In [6]:
from script_02_arima_sarima import carregar_serie_tds as carregar_serie_tds_02, rodar_stl_trend, rodar_sarima

s_tds = carregar_serie_tds_02()
linha_stl, stl_full = rodar_stl_trend(s_tds)
linha_sarima, res_sarima_full = rodar_sarima(s_tds)

tabela_series_temporais = pd.DataFrame([linha_stl, linha_sarima])[
    ['metodo', 'tendencia_mgL_ano', 'tendencia_pvalor', 'rmse_holdout', 'mae_holdout', 'r2_holdout']
]
tabela_series_temporais

  Melhor ordem SARIMA (AIC=1411.1 em treino): order=(0, 1, 2) seasonal_order=(0, 1, 1, 12) trend=None


,metodo,tendencia_mgL_ano,tendencia_pvalor,rmse_holdout,mae_holdout,r2_holdout
0,stl_tendencia_linear,2.917378,3.965230e-03,46.077543,37.414353,-0.350508
1,sarima,10.434279,1.771610e-46,48.377183,38.377663,-0.488675


O STL+tendência (2,92 mg/L/ano) fica na mesma ordem dos métodos clássicos da seção 3.a. O SARIMA, sem drift explícito na ordem escolhida por AIC, tem sua tendência de longo prazo estimada indiretamente a partir do caminho previsto (10,43 mg/L/ano) — mais acentuada, e com IC90% que já inclui valores negativos em +15 anos, sinalizando a fragilidade esperada de extrapolar um SARIMA muito além do histórico de treino (ver `Artigo/src/resultados.tex`).

### 3.c Modelos baseados em árvore — Random Forest, XGBoost, LightGBM (`script_03`, `script_04`)

Implementado (os três). Random Forest via `script_03`; XGBoost (GPU/CUDA, RTX 4060 Ti — confirmado nesta máquina) e LightGBM (CPU — o pacote pip não vem com build GPU) via `script_04`, reaproveitando as mesmas features de tempo/sazonalidade/lags e o mesmo esquema de previsão recursiva.

In [7]:
from script_03_random_forest_gridsearch import (
    carregar_serie_tds as carregar_serie_tds_03, construir_features, treinar_grid_search,
    prever_recursivo, FEATURES, HOLDOUT_MESES as HOLDOUT_MESES_03, HORIZONTES_ANOS as HORIZONTES_03,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

d_rf = carregar_serie_tds_03()
f_rf = construir_features(d_rf)
treino_rf, holdout_rf = f_rf.iloc[:-HOLDOUT_MESES_03], f_rf.iloc[-HOLDOUT_MESES_03:]

modelo_rf, params_rf = treinar_grid_search(treino_rf[FEATURES], treino_rf['TDS_mgL'])
pred_holdout_rf = modelo_rf.predict(holdout_rf[FEATURES])
rmse_rf = mean_squared_error(holdout_rf['TDS_mgL'], pred_holdout_rf) ** 0.5
r2_rf = r2_score(holdout_rf['TDS_mgL'], pred_holdout_rf)
print(f"Melhores hiperparametros: {params_rf}")
print(f"Holdout: RMSE={rmse_rf:.2f}  R2={r2_rf:.3f}")

modelo_rf_full, _ = treinar_grid_search(f_rf[FEATURES], f_rf['TDS_mgL'])
pontos_rf, baixos_rf, altos_rf = prever_recursivo(modelo_rf_full, d_rf, max(HORIZONTES_03) * 12)
for h in HORIZONTES_03:
    p = h * 12 - 1
    print(f"+{h}a: {pontos_rf[p]:.1f} mg/L  IC90% [{baixos_rf[p]:.1f}, {altos_rf[p]:.1f}]")

  Melhores hiperparametros (CV temporal, RMSE): {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 300}
Melhores hiperparametros: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 300}
Holdout: RMSE=40.99  R2=-0.069


  Melhores hiperparametros (CV temporal, RMSE): {'max_depth': 5, 'min_samples_leaf': 3, 'n_estimators': 300}


+10a: 704.3 mg/L  IC90% [665.5, 742.7]
+15a: 704.3 mg/L  IC90% [665.5, 742.7]
+20a: 704.3 mg/L  IC90% [665.5, 742.7]


O Random Forest teve o melhor RMSE em holdout entre os métodos até aqui, mas sua previsão recursiva **satura**: +10, +15 e +20 anos convergem para o mesmo valor (704,3 mg/L) — árvores não extrapolam além do range visto no treino, limitação estrutural conhecida (ver `plano_projeto_TDS.md` §3.c e `Artigo/src/resultados.tex`). Reportado explicitamente, não escondido.

In [8]:
from script_04_xgboost_lightgbm import (
    grid_search_xgb, grid_search_lgbm, treinar_quantis_xgb, treinar_quantis_lgbm,
    prever_recursivo as prever_recursivo_04, montar_linha,
    HOLDOUT_MESES as HOLDOUT_MESES_04, HORIZONTES_ANOS as HORIZONTES_04,
)

treino_04, holdout_04 = f_rf.iloc[:-HOLDOUT_MESES_04], f_rf.iloc[-HOLDOUT_MESES_04:]

modelo_xgb, params_xgb, usa_gpu = grid_search_xgb(treino_04[FEATURES], treino_04['TDS_mgL'])
pred_xgb = modelo_xgb.predict(holdout_04[FEATURES])
print(f"XGBoost holdout RMSE: {mean_squared_error(holdout_04['TDS_mgL'], pred_xgb) ** 0.5:.2f}  (GPU={usa_gpu})")

modelo_lgb, params_lgb = grid_search_lgbm(treino_04[FEATURES], treino_04['TDS_mgL'])
pred_lgb = modelo_lgb.predict(holdout_04[FEATURES])
print(f"LightGBM holdout RMSE: {mean_squared_error(holdout_04['TDS_mgL'], pred_lgb) ** 0.5:.2f}  (CPU)")

  XGBoost — melhores hiperparametros (GPU/CUDA): {'learning_rate': 0.03, 'max_depth': 7, 'n_estimators': 100}
XGBoost holdout RMSE: 49.52  (GPU=True)


  LightGBM — melhores hiperparametros (CPU): {'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 100}
LightGBM holdout RMSE: 44.42  (CPU)


Os três métodos baseados em árvore (RF, XGBoost, LightGBM) reproduzem a mesma limitação estrutural: nenhum capta a tendência de alta real do TDS ao extrapolar — as três tendências implícitas na previsão recursiva são negativas (-0,25 a -0,91 mg/L/ano), o oposto dos 3,7-3,9 mg/L/ano identificados pelos métodos estatísticos clássicos. O XGBoost ainda mostra oscilação não-monotônica entre horizontes (649,6 → 717,6 → 649,6 mg/L em +10/+15/+20 anos) — artefato de boosting extrapolando fora do range de treino. Ver `Artigo/src/resultados.tex` para a tabela e discussão completas.

### 3.d Prophet e regressão bayesiana (`script_05_prophet_bayesiano.py`)

Implementado. Os dois métodos escolhidos por exporem incerteza crescente com o horizonte de forma mais honesta que os métodos anteriores.

In [9]:
from script_05_prophet_bayesiano import carregar_serie_tds as carregar_serie_tds_05, rodar_prophet, rodar_bayesiano

d_05 = carregar_serie_tds_05()
linha_prophet, prev_full_prophet = rodar_prophet(d_05)
print("Prophet:", {k: round(v, 3) if isinstance(v, float) else v for k, v in linha_prophet.items() if 'metodo' in k or 'tendencia' in k or 'holdout' in k})

Importing plotly failed. Interactive plots will not work.


16:50:12 - cmdstanpy - INFO - Chain [1] start processing


16:50:12 - cmdstanpy - INFO - Chain [1] done processing


16:50:12 - cmdstanpy - INFO - Chain [1] start processing


16:50:12 - cmdstanpy - INFO - Chain [1] done processing


Prophet: {'metodo': 'prophet', 'tendencia_mgL_ano': np.float64(3.811), 'tendencia_pvalor': np.float64(0.0), 'tendencia_ic90_baixo': nan, 'tendencia_ic90_alto': nan, 'rmse_holdout': 48.966, 'mae_holdout': 40.473, 'r2_holdout': -0.525}


In [10]:
linha_bayes, _ = rodar_bayesiano(d_05)
print("Regressao bayesiana:", {k: round(v, 3) if isinstance(v, float) else v for k, v in linha_bayes.items() if 'metodo' in k or 'tendencia' in k or 'holdout' in k or 'prob' in k})

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


  Ajustando regressao bayesiana (treino, NUTS)...


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


...

Sequential sampling (2 chains in 1 job)


NUTS: [a, b, sigma]


Sampling 2 chains for 800 tune and 800 draw iterations (1_600 + 1_600 draws total) took 230 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


Regressao bayesiana: {'metodo': 'regressao_bayesiana', 'tendencia_mgL_ano': 3.742, 'tendencia_pvalor': 0.002, 'tendencia_ic90_baixo': 1.522, 'tendencia_ic90_alto': 5.861, 'rmse_holdout': 49.29, 'mae_holdout': 39.405, 'r2_holdout': -0.545, 'prob_tendencia_positiva': 0.999}


A regressão bayesiana confirma a tendência de alta (3,74 mg/L/ano, 99,88% de probabilidade posterior de tendência positiva) — praticamente idêntica à OLS, como esperado. O Prophet detecta corretamente a mesma tendência histórica (3,81 mg/L/ano, p<0,0001), mas sua extrapolação futura é **decrescente** (636,9 → 604,8 mg/L de +10 a +20 anos) — os changepoints automáticos do modelo capturaram uma desaceleração recente e a extrapolam, divergindo da tendência de longo prazo. Achado genuíno, não escondido — ver `Artigo/src/resultados.tex` para a discussão completa.

### 3.f Baselines obrigatórios (`script_08_baselines.py`)

Implementado. Naive, naive sazonal (m=12), ETS/Holt-Winters e Theta — o "piso" de MASE contra o qual toda a bateria (inclusive os métodos 3.a-3.d acima, agora recalculados com MASE/sMAPE/CV/backtest via `validacao_utils.py`) é julgada. Nenhum método mais sofisticado entra no relatório final sem vencer estes baselines.

In [ ]:
from script_08_baselines import carregar_serie_tds as carregar_serie_tds_08, carregar_usa_sazonalidade, rodar_naive, rodar_naive_sazonal, rodar_ets, rodar_theta
from validacao_utils import validar_metodo
import script_08_baselines as sb08

s_08 = carregar_serie_tds_08()
treino_08, holdout_08 = s_08.iloc[:-24], s_08.iloc[-24:]
sazonal_08 = carregar_usa_sazonalidade()

linhas_baseline = [rodar_naive(s_08), rodar_naive_sazonal(s_08), rodar_ets(s_08, sazonal_08), rodar_theta(s_08, sazonal_08)]
fit_predicts_08 = {
    'naive': sb08.fit_predict_naive, 'naive_sazonal': sb08.fit_predict_naive_sazonal,
    'ets_holt_winters': sb08.construir_fit_predict_ets(sazonal_08), 'theta': sb08.construir_fit_predict_theta(sazonal_08),
}
for linha in linhas_baseline:
    linha.update(validar_metodo(fit_predicts_08[linha['metodo']], s_08, treino_08, holdout_08))

tabela_baselines = pd.DataFrame(linhas_baseline)[['metodo', 'rmse_holdout', 'mase_holdout', 'cv_mase_media', 'forecast_20y']]
tabela_baselines

**Achado notável:** o baseline **naive** (repete o último valor observado) tem o **menor MASE de toda a bateria** no holdout de 24 meses (0,44 — melhor que todos os 10 métodos de 3.a-3.d). Isso não invalida os métodos mais sofisticados — eles capturam corretamente a tendência de alta de longo prazo, que o naive não captura (a previsão do naive é uma linha reta constante) — mas mostra que, no horizonte curto do holdout, a série tem flutuações que nenhum método (nem os sofisticados) prevê melhor que "nada muda". Ver `Artigo/src/resultados.tex` para a discussão completa, incluindo a distinção entre desempenho de curto prazo (MASE) e capacidade de captar a tendência de longo prazo (foco real dos objetivos do projeto).

### 3.g Métodos adicionais (`script_09` a `script_13`)

Implementados. Cinco métodos além da bateria original, testados sob o mesmo framework de validação honesta (`validacao_utils.py`): SVR + Gaussian Process (`script_09`), tendência+árvore no resíduo (`script_10`, corrige a saturação de RF/XGBoost), SARIMAX com Cloreto exógeno (`script_11`), híbrido SARIMA+Prophet (`script_12`) e LSTM leve (`script_13`, PyTorch). A tabela final (Seção 4) já inclui os 21 métodos — aqui só destacamos os achados mais notáveis, sem duplicar a lógica dos scripts.

In [ ]:
tabela_metodos_adicionais = pd.read_csv('resultados_comparacao.csv')
novos_metodos = ['svr', 'gaussian_process', 'detrend_rf', 'detrend_xgb', 'sarimax_cloreto', 'hibrido_arima_prophet', 'lstm']
cols = ['metodo', 'tendencia_mgL_ano', 'rmse_holdout', 'mase_holdout', 'cv_mase_media', 'r2_holdout', 'forecast_10y', 'forecast_20y']
tabela_metodos_adicionais[tabela_metodos_adicionais['metodo'].isin(novos_metodos)][cols].sort_values('mase_holdout')

**Detrend+RF** (MASE 0,44) quase empata com o naive e, ao contrário do RF original, sua previsão de longo prazo **cresce monotonicamente** (743,5 → 762,2 → 780,9 mg/L em +10/+15/+20a) — corrige diretamente a saturação já documentada. **SVR** é o primeiro método de toda a bateria com **R² positivo em holdout** (0,04). O **Gaussian Process** teve o pior desempenho de todos os 21 métodos (MASE 2,03) — converge para um length-scale curto e reverte à média, limitação conhecida de GP, reportada como obtida. O **Cloreto como regressor exógeno** (SARIMAX) tem coeficiente significativo mas não melhora o holdout sobre o SARIMA univariado. O **híbrido SARIMA+Prophet** supera os dois componentes isolados no RMSE. O **LSTM não venceu o naive** (MASE 0,69 vs 0,44) — confirma o precedente do material de apoio de que deep learning leve tende a perder para métodos clássicos/boosting em séries ambientais mensais curtas. Ver `Artigo/src/resultados.tex` §Métodos adicionais para a discussão completa.

## 4. Tabela comparativa final (`resultados_comparacao.csv`)

Todos os 21 métodos da bateria (10 da Seção 3.a-3.d + 4 baselines da 3.f + 7 métodos adicionais da 3.g) foram executados, todos com MASE/sMAPE/CV expansiva (5 folds)/backtest de origem móvel via `validacao_utils.py`. A tabela abaixo é lida diretamente do CSV consolidado — nenhum número é digitado à mão.

In [ ]:
tabela_final = pd.read_csv('resultados_comparacao.csv')
colunas_exibir = ['metodo', 'tendencia_mgL_ano', 'mase_holdout', 'cv_mase_media', 'r2_holdout', 'forecast_10y', 'forecast_15y', 'forecast_20y']
tabela_final[colunas_exibir].sort_values('mase_holdout')

Os cinco métodos com ajuste global à série inteira (Mann-Kendall/Sen, Theil-Sen, OLS, STL+tendência, regressão bayesiana) convergem para uma tendência de alta consistente (2,9-3,9 mg/L/ano, p<0,01). O detrend+RF (Seção 3.g) corrige a limitação de extrapolação dos métodos de árvore originais. SARIMA e Prophet isolados divergem em direções opostas na extrapolação; o híbrido atenua essa divergência. Apenas 1 dos 21 métodos (SVR) tem R² positivo no holdout de 24 meses — a série tem flutuações de curto prazo que a maioria dos métodos não captura bem. A evidência mais robusta é a **convergência** dos três finalistas recomendados (Seção 4.3) em torno de ~760-800 mg/L em +20 anos, não a previsão pontual de um único modelo. Ver `Artigo/src/resultados.tex` §Síntese comparativa para a discussão completa.

## 4.1 Diagnóstico de resíduos dos candidatos mais fortes (`script_14_diagnostico_residuos.py`)

Implementado. Ljung-Box (autocorrelação), Shapiro-Wilk (normalidade) e ARCH (heterocedasticidade) sobre os resíduos *in-sample* de 5 candidatos: OLS, regressão bayesiana (tratada como equivalente à OLS — mesmo modelo linear-gaussiano), Detrend+RF, SARIMA e híbrido SARIMA+Prophet.

In [ ]:
import json
diag = json.load(open('diagnostico_residuos_resultados.json', encoding='utf-8'))
pd.DataFrame(diag)[['metodo', 'ljung_box_pvalor', 'ljung_box_ok', 'shapiro_pvalor', 'shapiro_ok', 'arch_pvalor', 'arch_ok']]

**Detrend+RF é o único candidato sem autocorrelação residual nem heterocedasticidade condicional detectável** (só falha normalidade — caudas mais pesadas, comum em árvores). Os quatro métodos com forma funcional linear/estocástica explícita falham em pelo menos 2 dos 3 testes — reforça que nenhum método individual captura completamente a dinâmica de curto prazo da série.

## 4.2 Comparação com a literatura

Schwabe et al. (2020) — 34 estações do sul da Califórnia, 2013-2017 — e Wolfand et al. (2022) — mesma bacia do rio Los Angeles onde a LAGWRP descarrega — encontram a mesma direção de efeito (conservação/reúso de água → maior salinidade/TDS) que este trabalho identificou de forma independente. A comparação é qualitativa: o texto completo de nenhum dos dois foi acessível nesta sessão (ambos pagos), então nenhuma magnitude numérica desses estudos é reproduzida sem verificação direta — regra de governança deste projeto. Ver `Artigo/src/resultados.tex` §Comparação com a literatura e `Artigo/src/trabalhos-relacionados.tex` para a discussão completa.

## 4.3 Síntese final e finalistas recomendados (`script_15_sintese_final.py`)

Implementado. Três finalistas complementares (não um vencedor único): **regressão bayesiana** (incerteza honesta), **Detrend+RF** (melhor MASE entre os que captam tendência + resíduos mais limpos) e **híbrido SARIMA+Prophet** (cenário mais cauteloso, IC90 mais largo).

In [ ]:
tabela_sintese = pd.read_csv('tabela_sintese_final.csv')
tabela_sintese[tabela_sintese['finalista']][['metodo', 'mase_holdout', 'rmse_holdout', 'forecast_10y', 'forecast_15y', 'forecast_20y']]

Os três finalistas convergem para ~760-800 mg/L em +20 anos apesar de mecanismos de extrapolação completamente diferentes — essa convergência é mais informativa que a previsão pontual de qualquer um isoladamente. O IC90 do híbrido é notavelmente mais largo (herdado da componente SARIMA), refletindo de forma honesta a incerteza real de extrapolar ~15 anos de histórico para +20 anos à frente.

## 5. Conclusão / discussão

Os quatro objetivos do projeto foram endereçados com uma bateria de 21 métodos, todos com validação temporal honesta (MASE, CV expansiva, backtest de origem móvel). TDS: tendência de alta consistente e significativa (~3-4 mg/L/ano), com três finalistas complementares convergindo para ~760-800 mg/L em +20 anos. Correlação TDS-Amônia positiva e significativa mesmo destendenciada (r=0,175, p=0,018) — consistente com a hipótese de inibição biológica por salinidade. TDS-BOD sem correlação detectável em nenhum dos três tratamentos de ND (incluindo o ROS/Helsel, o mais bem embasado estatisticamente), provavelmente por falta de variação real na série de BOD. Cloreto como regressor exógeno adicional não melhora a previsão de TDS de forma prática. Os achados são compatíveis com o mecanismo de conservação de água descrito por Schwabe et al. (2020) e Wolfand et al. (2022) — comparação qualitativa, sem reproduzir magnitudes não verificadas. Interpretação completa, limitações e trabalhos futuros em `Artigo/src/conclusao.tex`; contexto da literatura em `Artigo/src/trabalhos-relacionados.tex`.

## 6. Tratamento de dados robusto (prompt_tratamento_e_metodos.md)

### 6.1 Reconstrução da vazão (`script_16_reconstrucao_vazao.py`)

Implementado. `lb/day = mg/L × vazão(MGD) × 8,34` permite reconstruir a vazão do efluente a partir dos dados já existentes (concentração + carga mássica do mesmo parâmetro). Validado em duas frentes: plausibilidade contra a capacidade nominal (20 MGD) e consistência cruzada entre TDS/Cloreto/Amônia/BOD (medidos na mesma amostra física — se a identidade for real, a vazão derivada de cada um deve coincidir).

In [ ]:
import json
vazao = json.load(open('vazao_reconstruida_resultados.json', encoding='utf-8'))
print(f"Correlação média vazão(TDS) com Cloreto/Amônia/BOD: {vazao['correlacao_media_tds_outros']:.3f}")
print(f"Identidade se sustenta: {vazao['identidade_sustenta']}")
pd.DataFrame(vazao['plausibilidade_por_parametro']).T[['n', 'media', 'mediana', 'pct_capacidade_nominal']]

**A identidade se sustenta**: vazão derivada de TDS correlaciona 0,997-0,998 com Cloreto/Amônia (0,87 com BOD, mais ruidoso mas ainda forte), vazão média ~9,5 MGD (~47% da capacidade nominal de 20 MGD) — plausível e consistente. Habilita os métodos 3.f.1-3.f.3 (WRTDS, balanço de massa, cenários) **na Etapa 3, aguardando aprovação do usuário**.

### 6.2 Matriz de sensibilidade no tratamento de dados (`script_17_matriz_sensibilidade.py`)

Implementado. 9 decisões de pré-processamento testadas ANTES de fixar qualquer tratamento padrão, medindo o impacto na tendência de TDS (ou na correlação TDS-BOD, item 1). **Resultados aguardando decisão do usuário — nenhum tratamento foi fixado como padrão.**

In [ ]:
matriz = pd.read_csv('matriz_sensibilidade_resultados.csv')
matriz[['item', 'variante', 'metrica', 'valor_metrica', 'pvalor', 'n']]

**Achados que qualificam a conclusão central do projeto (tendência de alta de TDS estatisticamente significativa):**
- **Item 2 (quebra EFF-001→EFF-001A):** a tendência cai de 3,91 mg/L/ano (p=0,0056, série completa) para 0,84 mg/L/ano (p=0,59, **não significativo**) quando restrita só ao período EFF-001A (2012-2026, 170 dos 182 meses). Os 12 meses iniciais sob o código antigo têm peso desproporcional na tendência da série completa.
- **Item 3 (mudança de MDL):** 3 das 4 transições de MDL coincidem com mudança estatisticamente significativa (Mann-Whitney) no nível médio de TDS — possível confusão entre mudança de método analítico e tendência real.
- **Item 5 (agregação anual):** nenhuma das 4 variantes (média/mediana × ano civil/hidrológico, 15 pontos anuais) é estatisticamente significativa (p entre 0,30 e 0,44) — poder estatístico bem menor com 15 pontos, mas o padrão é notável.
- **Item 8 (autocorrelação no MK):** o p-valor da tendência **muda de significativo (p=0,0056) para não significativo** sob duas das quatro correções de autocorrelação testadas (Hamed-Rao p=0,182; pre-whitening p=0,417), mas continua significativo nas outras duas (trend-free pre-whitening p=0,0001; Seasonal Kendall p=0,0038).

**Achados que reforçam a robustez:** item 4 (reagregação das amostras brutas bate exatamente com o "Monthly Average" pronto — diferença média 0,0 mg/L); item 6 (tendência estável entre 3,6-3,9 mg/L/ano com/sem outliers, todas p<0,01); item 7 (nenhum mês faltante); item 9 (mesmo p-valor em escala bruta e log, como esperado — Kendall's tau é invariante a transformação monotônica).

**Conclusão honesta:** a tendência de alta de TDS não é um artefato óbvio de outliers ou de reagregação (itens 4, 6, 9 dão suporte), mas **é sensível** à forma de tratar a transição EFF-001→EFF-001A, à mudança de MDL, e à correção de autocorrelação — 3 sinais convergentes de que a "significância" da tendência mensal pode estar inflada. Isto **não invalida** a tendência de alta (o sinal de alta persiste em quase todas as variantes, incluindo as correções mais rigorosas TFPW e Seasonal Kendall), mas justifica reportar a tendência com uma faixa de incerteza mais ampla do que "p<0,01" sugere isoladamente. Ver `matriz_sensibilidade_resultados.csv` para a tabela completa. **Decisão de quais tratamentos adotar como padrão pendente — aguardando aprovação do usuário antes de prosseguir para a Etapa 3 (bateria de métodos ampliada).**